# Demo: Fine-Tuning a Sentiment Classifier on SageMaker

This notebook demonstrates fine-tuning **DistilBERT** on SageMaker using a custom software product review dataset.

The model learns to classify reviews as:
- `0` — Negative
- `1` — Positive

This is a common real-world use case: companies fine-tune a general-purpose sentiment model on their own domain-specific vocabulary so it understands product-specific language accurately.

This notebook walks through:
- Setting up the environment
- Uploading the training dataset
- Fine-tuning DistilBERT with custom hyperparameters
- Deploying and testing the fine-tuned classifier
- Cleaning up all resources

> **Training time:** ~5 minutes on `ml.m5.xlarge`  
> **Estimated cost:** under $0.10 if you clean up promptly


## 1. Configuration

In [ ]:
# --- Tuneable config ---
MODEL_ID           = 'huggingface-tc-distilbert-base-uncased'
INFERENCE_INSTANCE = 'ml.m5.xlarge'
LOCAL_DATASET      = 'data.csv'
S3_KEY_PREFIX      = 'sentiment-finetune-data'

# Custom hyperparameters
EPOCHS        = 3
LEARNING_RATE = 2e-5
BATCH_SIZE    = 16

## 2. Setup

In [ ]:
# Install the correct AWS SageMaker Python SDK (v2.x)
!pip install 'sagemaker>=2.0,<3.0' --quiet

# Restart kernel so the newly installed package is available
import sys
if 'sagemaker' in sys.modules:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Import libraries
import boto3
import sagemaker
from sagemaker.session import Session

session = Session()
role    = sagemaker.get_execution_role()
bucket  = session.default_bucket()
region  = session.boto_region_name

print(f'Region : {region}')
print(f'Bucket : {bucket}')
print(f'Role   : {role}')

## 3. Upload Dataset

The dataset contains 400 software product reviews labelled as positive (1) or negative (0).
Each row has the integer label in the first column and the review text in the second column — no header.
Binary classification (2 classes) gives the model the best chance of learning accurately from this dataset size.

In [ ]:
# Upload dataset to S3
# The container requires the file to be named data.csv and the S3 path must end with /
from sagemaker.s3 import S3Uploader

s3_data_path = f's3://{bucket}/{S3_KEY_PREFIX}'
S3Uploader.upload(LOCAL_DATASET, s3_data_path)
s3_path = s3_data_path + '/'
print('Uploaded training data to:', s3_path)

## 4. Fine-Tune with JumpStart

DistilBERT is fine-tuned on the custom dataset using the hyperparameters defined in the config cell.
The model learns to associate domain-specific software review language with the correct sentiment label.

In [ ]:
# Load JumpStart estimator for DistilBERT text classification
from sagemaker.jumpstart.estimator import JumpStartEstimator
from sagemaker import instance_types

# Look up the default supported training instance type for this model
default_instance = instance_types.retrieve_default(
    model_id=MODEL_ID,
    model_version='*',
    scope='training'
)
print('Default training instance type:', default_instance)

estimator = JumpStartEstimator(
    model_id=MODEL_ID,
    role=role,
    instance_type=default_instance
)

In [ ]:
# Configure custom hyperparameters
estimator.set_hyperparameters(
    epochs=EPOCHS,
    adam_learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE
)

In [ ]:
# Start training job (blocks until complete — expect ~5 minutes)
estimator.fit({'training': s3_path}, wait=True)

## 5. Verify Training

In [ ]:
# Check training job status and final accuracy metric
sm_client = boto3.client('sagemaker', region_name=region)
job = sm_client.describe_training_job(
    TrainingJobName=estimator.latest_training_job.name
)
print('Training job status   :', job['TrainingJobStatus'])
print('Model artifact S3 path:', job['ModelArtifacts']['S3ModelArtifacts'])

metrics = job.get('FinalMetricDataList', [])
if metrics:
    for m in metrics:
        print(f"{m['MetricName']}: {m['Value']:.4f}")
else:
    print('No metrics available')

## 6. Deploy

In [ ]:
# Deploy the fine-tuned classifier
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type=INFERENCE_INSTANCE
)
print('Endpoint name:', predictor.endpoint_name)

## 7. Test

Test the fine-tuned model with software review sentences it hasn't seen before.
Each prediction returns a label (0 = Negative, 1 = Positive) and a confidence score.

In [ ]:
# Test the fine-tuned sentiment classifier
# Label map: 0 = Negative, 1 = Positive
label_map = {0: 'Negative', 1: 'Positive'}

test_reviews = [
    'The software keeps freezing during our daily standups.',
    'The new release has made everything noticeably faster.',
    'We lost a week of data after a failed migration with no support.',
    'The API documentation is clear and the endpoints work as described.',
    'The product has made our team significantly more productive.',
    'We have submitted the same bug report four times with no response.',
]

for review in test_reviews:
    response = predictor.predict(review.encode('utf-8'), initial_args={'ContentType': 'application/x-text'})
    probabilities = response.get('probabilities', [])
    predicted_label = int(probabilities.index(max(probabilities)))
    sentiment = label_map.get(predicted_label, str(predicted_label))
    confidence = max(probabilities) * 100
    print(f'Review    : {review}')
    print(f'Sentiment : {sentiment} ({confidence:.1f}% confidence)\n')

## 8. Cleanup
**Run this cell as soon as you finish testing to avoid ongoing charges.**

In [ ]:
# Delete ALL resources to avoid ongoing charges
s3_client = boto3.client('s3', region_name=region)

# 1. Delete the endpoint
try:
    predictor.delete_endpoint()
    print('✓ Endpoint deleted')
except Exception as e:
    print(f'Endpoint deletion skipped: {e}')

# 2. Delete uploaded dataset from S3
try:
    s3_client.delete_object(Bucket=bucket, Key=f'{S3_KEY_PREFIX}/{LOCAL_DATASET}')
    print('✓ S3 dataset deleted')
except Exception as e:
    print(f'S3 dataset deletion skipped: {e}')

# 3. Delete training job output artifacts from S3
try:
    training_job_name = estimator.latest_training_job.name
    prefix = f'{training_job_name}/output/'
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    objects = response.get('Contents', [])
    if objects:
        s3_client.delete_objects(
            Bucket=bucket,
            Delete={'Objects': [{'Key': obj['Key']} for obj in objects]}
        )
        print(f'✓ Training artifacts deleted ({len(objects)} objects)')
    else:
        print('No training artifacts found in S3')
except Exception as e:
    print(f'Training artifact deletion skipped: {e}')

print('\nCleanup complete. Verify in the AWS Console that no endpoints or models remain.')